# Stage 10 MVP — 约束规则下观察模型真实输出

> **用途**：输入问题 → 强约束流水线 → 查看模型**完整答案**与安检结果。  
> 与 `constraint-hallucination.ipynb`（规则/mock 演示）不同，本页面向试用与观察原文。

### 推荐操作顺序

1. **§1 选模式**（决定后面要不要加载全量检索）
2. **§2 环境 + 流水线**（按 `MODE` **针对性构建**）
3. **§3 写问题**
4. **§4 执行** → 看完整答案
5. **§5 可选保存**

| 你想… | 操作 |
|--------|------|
| 换问题（同模式） | 只改 §3，再跑 §4 |
| 换模式 | 改 §1 → **必须重跑 §2**（重建流水线）→ 再 §3/§4 |

## §1. 选择运行模式（最先跑；只改 `MODE`）

`MODE` 会传给 §2，决定检查哪些资源、是否加载 610 万全量检索。

### 三种模式对比

| `MODE` | 真 Ollama | 文献从哪来 | §2 会做什么 | 适合什么 | 耗时 |
|--------|-----------|------------|-------------|----------|------|
| **`live`** | ✅ | **全量检索**按问题召回 | 检查 corpus + Ollama；`from_mode("full")` 加载 Chroma/BM25 | 日常试用「真实 RAG + 约束」 | **数分钟/条**；冷启动加载索引更久 |
| **`fixture`** | ✅ | **固定假文献**（对抗用例 / 空列表） | 只检查 Ollama；**不**加载全量检索；真 LLM + 空检索桩 | OOD/诱导/假引用等陷阱；想控制「模型只能看到这些段落」 | 通常比 live 快（省检索） |
| **`demo`** | ❌ | 假文献或空 | 不依赖 Ollama/全库；脚本 LLM | 没开 Ollama 时看打印格式 | 秒级；**答案不是模型输出** |

### 怎么选（决策树）

```text
想看 deepseek-r1 真实回答？
  ├─ 否 / Ollama 没开 → MODE = "demo"
  └─ 是
       ├─ 测拒答/诱导/假引用等陷阱 → MODE = "fixture"（§3 设 PRESET）
       └─ 测普通问题、看检索+生成 → MODE = "live"（§3 只写 USER_QUERY，PRESET=None）
```

### 为何模式要放在环境之前？

- `live` 才值得付「加载全量索引」的成本。
- `fixture` 若也 `from_mode("full")`，会白白冷启 BM25/Chroma，却不用检索。
- 因此：**先定 MODE → §2 按 MODE 构建**，换模式必须重跑 §2。

In [1]:
# ==============================
# 【模式选择】只改这一行
# 可选: "live" | "fixture" | "demo"
# ==============================
MODE = "live"

SAVE_SAMPLE = True  # §5 是否保存本次完整答案 JSON

# 流水线工程参数（一般不用改）
MAX_RETRIES = 1
MAX_CONTEXT_TOKENS = 1200
SKIP_EVIDENCE_EVAL = True
SKIP_CRITICAL_REVIEW = True
LLM_TIMEOUT = 300.0

assert MODE in ("live", "fixture", "demo"), f"MODE 非法: {MODE!r}"
print("已选择 MODE =", repr(MODE))
print("下一步: 运行 §2（将按该模式针对性构建环境与流水线）")
if MODE == "live":
    print("→ §2 将检查全量语料 + Ollama，并加载 RetrievalPipeline.from_mode('full')")
elif MODE == "fixture":
    print("→ §2 将只检查 Ollama，构建「真 LLM + 无检索」流水线（不加载 610 万索引）")
else:
    print("→ §2 将构建脚本 demo 流水线（不调用 Ollama）")

已选择 MODE = 'live'
下一步: 运行 §2（将按该模式针对性构建环境与流水线）
→ §2 将检查全量语料 + Ollama，并加载 RetrievalPipeline.from_mode('full')


## §2. 环境初始化 + 构建流水线（读取上方 `MODE`）

本单元**依赖 §1 的 `MODE`**：

| `MODE` | 资源检查 | 构建内容 |
|--------|----------|----------|
| `live` | corpus ready + Ollama | `ConstrainedGenerationPipeline.from_mode("full")` |
| `fixture` | 仅 Ollama | 真 `LLMGenerator` + `NoRetrieval` + `ContextAssembler` |
| `demo` | 无硬性依赖 | 脚本 LLM + `NoRetrieval` |

换过 `MODE` 后请**重新运行本单元**。

In [2]:
from pathlib import Path
import sys
import json
import time

CWD = Path.cwd().resolve()
STAGE10 = CWD.parent if CWD.name == "notebooks" else CWD
SRC = STAGE10 / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

# 必须先 bootstrap（挂 05–10），再 import 07/08 模块。
# 强制 reload：避免 Jupyter 仍缓存「没有 OLLAMA_*」的旧 bootstrap。
import importlib

import bootstrap as _bootstrap_mod

importlib.reload(_bootstrap_mod)
from bootstrap import OLLAMA_BASE_URL, OLLAMA_MODEL, bootstrap_paths

paths = bootstrap_paths(STAGE10)

# constrained_pipeline 也可能被内核缓存，一并 reload
import constrained_pipeline as _cp_mod

importlib.reload(_cp_mod)

from config import DEFAULT_CONFIG, Stage10Config
from resources import check_full_corpus_resources, probe_ollama
from adversarial_eval import load_adversarial_cases, default_cases_path
from constrained_pipeline import ConstrainedGenerationPipeline
from context_assembler import ContextAssembler  # 07：依赖 bootstrap 后的 sys.path

cfg = Stage10Config(max_retries=MAX_RETRIES)
print("OLLAMA_MODEL:", OLLAMA_MODEL, "| base:", OLLAMA_BASE_URL)

print("MODE (from §1):", MODE)
print("stage10:", paths["stage10"])
print("defaults: retrieval_mode=", DEFAULT_CONFIG.retrieval_mode,
      "| refusal:", DEFAULT_CONFIG.refusal_en[:48], "...")

CASES = {c.id: c for c in load_adversarial_cases(default_cases_path(STAGE10))}
PRESET_IDS = {
    "ood": "ood_2025_fda",
    "induce": "induce_fabrication_side_effects",
    "terminology": "terminology_tavr",
    "fake_cite": "fake_citation_99",
    "normal": "normal_metformin",
}


class NoRetrieval:
    def run(self, query):
        return {"query": query, "retrieval": {"fused": []}, "reranked": []}


def _build_demo_pipe():
    class DemoLLM:
        def generate(self, prompt, **kwargs):
            blob = (kwargs.get("system_prompt") or "") + prompt
            if "Draft:\n" not in blob and "CORRECTION REQUIRED" not in blob:
                return "Draft."
            return (
                "**Answer:** Demo (scripted) answer [1].\n\n"
                "**Evidence Summary:**\n- See [1]\n"
            )

    return ConstrainedGenerationPipeline(
        retrieval_pipeline=NoRetrieval(),
        context_assembler=ContextAssembler(tokenizer_name=None),
        llm_generator=DemoLLM(),
        config=cfg,
        skip_evidence_eval=True,
        skip_critical_review=True,
        run_optional_eval=True,
    )


def _build_fixture_pipe():
    """真 Ollama，不加载全量检索索引。"""
    from llm_generator import LLMGenerator

    llm = LLMGenerator(
        model_name=OLLAMA_MODEL,
        base_url=OLLAMA_BASE_URL,
        timeout=LLM_TIMEOUT,
    )
    return ConstrainedGenerationPipeline(
        retrieval_pipeline=NoRetrieval(),
        context_assembler=ContextAssembler(tokenizer_name=None),
        llm_generator=llm,
        config=cfg,
        skip_evidence_eval=SKIP_EVIDENCE_EVAL,
        skip_critical_review=SKIP_CRITICAL_REVIEW,
        max_context_tokens=MAX_CONTEXT_TOKENS,
        run_optional_eval=True,
    )


def _build_live_pipe():
    return ConstrainedGenerationPipeline.from_mode(
        DEFAULT_CONFIG.retrieval_mode,
        config=cfg,
        skip_evidence_eval=SKIP_EVIDENCE_EVAL,
        skip_critical_review=SKIP_CRITICAL_REVIEW,
        max_context_tokens=MAX_CONTEXT_TOKENS,
        run_optional_eval=True,
        llm_timeout=LLM_TIMEOUT,
    )


PIPELINE_KIND = MODE  # 成功构建后与 MODE 对齐；失败则可能降级
corpus = {"ready": False, "status": "skipped"}
ollama = {"ok": False}

if MODE == "demo":
    pipe = _build_demo_pipe()
    print("\n✅ pipeline: demo (scripted, no Ollama / no corpus load)")

elif MODE == "fixture":
    ollama = probe_ollama()
    print("ollama:", json.dumps(ollama, ensure_ascii=False))
    if not (ollama.get("ok") and ollama.get("model_available")):
        print("\n⚠️ Ollama 不可用 → 降级 demo")
        pipe = _build_demo_pipe()
        PIPELINE_KIND = "demo"
    else:
        pipe = _build_fixture_pipe()
        print("\n✅ pipeline: fixture (real LLM, NoRetrieval — 未加载全量索引)")

elif MODE == "live":
    corpus = check_full_corpus_resources(STAGE10)
    print("corpus ready:", corpus.get("ready"), "|", corpus.get("status"))
    ollama = probe_ollama()
    print("ollama:", json.dumps(ollama, ensure_ascii=False))
    ollama_ok = bool(ollama.get("ok") and ollama.get("model_available"))
    if not corpus.get("ready"):
        raise RuntimeError(
            "MODE=live 需要全量语料 ready。请检查 chroma/chunks/BM25，或改用 MODE='fixture' / 'demo'。"
        )
    if not ollama_ok:
        print("\n⚠️ Ollama 不可用 → 降级 demo（无法真检索+生成）")
        pipe = _build_demo_pipe()
        PIPELINE_KIND = "demo"
    else:
        print("\n⏳ 正在加载全量检索流水线（可能较慢）...")
        pipe = _build_live_pipe()
        print("✅ pipeline: live (full retrieval + real LLM)")

else:
    raise ValueError(MODE)

print("PIPELINE_KIND:", PIPELINE_KIND)
print("is_real_model:", PIPELINE_KIND in ("live", "fixture"))

OLLAMA_MODEL: deepseek-r1:7b | base: http://127.0.0.1:11434
MODE (from §1): live
stage10: D:\谷歌\10 强约束规则开发与幻觉抑制
defaults: retrieval_mode= full | refusal: Based on the provided literature, this question  ...
corpus ready: True | ready
ollama: {"ok": true, "base_url": "http://127.0.0.1:11434", "model_requested": "deepseek-r1:7b", "model_available": true, "models_preview": ["deepseek-r1:7b"]}

⏳ 正在加载全量检索流水线（可能较慢）...
✅ pipeline: live (full retrieval + real LLM)
PIPELINE_KIND: live
is_real_model: True


## §3. 写问题（测试入口 —— 改题只动本单元）

### 两种提问方式（二选一）

| 方式 | 你改什么 | 推荐搭配的 `MODE` |
|------|----------|-------------------|
| **A. 自由提问** | 只改 **`USER_QUERY`**，保持 **`PRESET = None`** | `live` |
| **B. 快捷陷阱题** | 只改 **`PRESET`** | `fixture`（也可用 live 流水线但会强制用 fixture 文献） |

> `PRESET` 非空时会**覆盖** `USER_QUERY`，并自动带上该用例的 `fixture_chunks`。

### 快捷题 `PRESET`

| 取值 | 意图 | 期望观察 |
|------|------|----------|
| `None` | 使用 `USER_QUERY` | 自由提问 |
| `"ood"` | 超知识库 / 空文献 | 固定拒答句 |
| `"induce"` | 诱导编造副作用 | 拒答或不编造剂量/% |
| `"terminology"` | TAVR 无全称 | 首次写全称 |
| `"fake_cite"` | 诱导引 `[99]` | 最终无非法编号 |
| `"normal"` | metformin 对照 | 正常作答 + 合法引用 |

查询语言：工程默认 **英文**（与 05–09 一致）。

In [3]:
# ============================================================
# 【写问题】测试者主要改这里
# ============================================================

# —— 方式 A：自由提问（PRESET 必须为 None）——
USER_QUERY = "metformin cardiovascular effects"

# —— 方式 B：快捷题（非 None 则忽略 USER_QUERY）——
# None | "ood" | "induce" | "terminology" | "fake_cite" | "normal"
PRESET = None

# ============================================================

fixture_chunks = None
active_query = USER_QUERY.strip()
preset_meta = None

if PRESET is not None:
    cid = PRESET_IDS.get(PRESET)
    if not cid or cid not in CASES:
        raise ValueError(f"PRESET 非法: {PRESET!r}；可选 {list(PRESET_IDS)} 或 None")
    case = CASES[cid]
    active_query = case.query
    fixture_chunks = case.fixture_chunks  # 可能是 []
    preset_meta = {
        "id": case.id,
        "case_type": case.case_type,
        "expected_boundary_hit": case.expected_boundary_hit,
        "expected_behavior": case.expected_behavior,
        "notes": case.notes,
    }

print("─" * 60)
print("MODE:", MODE, "| PIPELINE_KIND:", PIPELINE_KIND)
print("本轮将提问:")
print(active_query)
print("─" * 60)
print("PRESET:", PRESET)
if preset_meta:
    print("preset 元信息:")
    print(json.dumps(preset_meta, ensure_ascii=False, indent=2))
print(
    "fixture_chunks:",
    "未绑定（live 将检索）"
    if fixture_chunks is None
    else f"{len(fixture_chunks)} 段固定文献",
)

if MODE == "fixture" and fixture_chunks is None:
    print("⚠️ MODE=fixture 但 PRESET=None → 将使用空文献；建议设 PRESET。")
if MODE == "live" and PRESET is not None:
    print("NOTE: live 流水线 + PRESET → 本轮仍用 fixture 文献（不跑检索），避免冲掉陷阱。")
    print("      要全量检索：设 PRESET=None，只改 USER_QUERY。")
if MODE == "demo":
    print("NOTE: demo 模式答案为脚本，不是 deepseek-r1。")

────────────────────────────────────────────────────────────
MODE: live | PIPELINE_KIND: live
本轮将提问:
metformin cardiovascular effects
────────────────────────────────────────────────────────────
PRESET: None
fixture_chunks: 未绑定（live 将检索）


## §4. 执行：查看模型完整输出

重点看 **`MODEL ANSWER (full)`**。  
- `MODE=live` 且 `PRESET=None` → 全量检索  
- 其余（`fixture` / 有 PRESET / `demo`）→ `fixture_chunks`（可为空）

In [4]:
# live + 无 PRESET → 真检索；否则固定 context
use_fixture = not (MODE == "live" and PIPELINE_KIND == "live" and fixture_chunks is None)

t0 = time.perf_counter()
if use_fixture:
    chunks = [] if fixture_chunks is None else fixture_chunks
    result = pipe.run(active_query, fixture_chunks=chunks)
else:
    result = pipe.run(active_query)
elapsed = round(time.perf_counter() - t0, 2)

is_real = PIPELINE_KIND in ("live", "fixture")
result["mvp_meta"] = {
    "mode": MODE,
    "pipeline_kind": PIPELINE_KIND,
    "elapsed_seconds": elapsed,
    "preset": preset_meta,
    "used_fixture_chunks": use_fixture,
    "is_real_model": is_real,
}

print("=" * 72)
print("QUERY")
print("=" * 72)
print(active_query)
print()
print(
    f"MODE={MODE} | PIPELINE_KIND={PIPELINE_KIND} | "
    f"real_model={is_real} | use_fixture={use_fixture} | "
    f"elapsed={elapsed}s | retry={result.get('retry_count')} | repaired={result.get('repaired')}"
)
print()
print("=" * 72)
print("MODEL ANSWER (full)")
print("=" * 72)
print(result.get("answer") or "(empty)")
print()
print("=" * 72)
print("CONSTRAINT CHECKS")
print("=" * 72)
print(json.dumps(result.get("constraint_checks"), indent=2, ensure_ascii=False))
if result.get("optional_evaluation"):
    print()
    print("optional_evaluation:", json.dumps(result["optional_evaluation"], ensure_ascii=False))
print()
print("=" * 72)
print("LABELED CONTEXT PREVIEW")
print("=" * 72)
print(result.get("labeled_context_preview") or "(empty)")
print()
srcs = result.get("sources") or []
print(f"sources (count={len(srcs)}):")
print(json.dumps(srcs, indent=2, ensure_ascii=False)[:1500])

[Reranker] 加载 BAAI/bge-reranker-base（首次运行需下载 ~1.1GB，请耐心等待）...
[Reranker] 就绪，device=cuda
QUERY
metformin cardiovascular effects

MODE=live | PIPELINE_KIND=live | real_model=True | use_fixture=False | elapsed=236.86s | retry=0 | repaired=False

MODEL ANSWER (full)
**Core Answer:**

Metformin has demonstrated various cardiovascular benefits, including enhanced myocardial energy metabolism and reduced risks of cardiovascular events such as stroke, coronary artery disease (CAD), and heart failure. Its effectiveness may vary depending on the patient's condition.

**Evidence Summary:**

1. **[1]** Metformin enhances myocardial energy metabolism by activating AMPK and regulating lipid and glucose metabolism, leading to improved nitric oxide (NO) bioavailability, reduced interstitial fibrosis, decreased deposition of advanced glycation end-products (AGEs), and inhibition of myocardial cell apoptosis. These mechanisms contribute to reducing cardiac remodeling and hypertrophy, thereby improving l

## §5. 可选：保存本次样例

写入 `outputs/samples/mvp_model_output_*.json`（带时间戳），不覆盖 mock 报告。是否保存由 §1 的 `SAVE_SAMPLE` 控制。

In [5]:
if SAVE_SAMPLE:
    out_dir = STAGE10 / "outputs" / "samples"
    out_dir.mkdir(parents=True, exist_ok=True)
    stamp = time.strftime("%Y%m%d_%H%M%S")
    tag = (PRESET or "custom").replace(" ", "_")
    out_path = out_dir / f"mvp_model_output_{tag}_{stamp}.json"
    slim = {
        "mvp_meta": result.get("mvp_meta"),
        "query": result.get("query"),
        "answer": result.get("answer"),
        "sources": result.get("sources"),
        "constraint_checks": result.get("constraint_checks"),
        "retry_count": result.get("retry_count"),
        "repaired": result.get("repaired"),
        "optional_evaluation": result.get("optional_evaluation"),
        "labeled_context_preview": result.get("labeled_context_preview"),
        "timestamp": result.get("timestamp"),
    }
    out_path.write_text(json.dumps(slim, indent=2, ensure_ascii=False), encoding="utf-8")
    print("saved:", out_path)
    if not slim.get("mvp_meta", {}).get("is_real_model"):
        print("⚠️ is_real_model=False（demo/降级），报告引用时请注明。")
else:
    print("SAVE_SAMPLE=False — 未保存")

saved: D:\谷歌\10 强约束规则开发与幻觉抑制\outputs\samples\mvp_model_output_custom_20260716_090052.json
